In [ ]:
#@title 1. Генерация озвучки PlayFit — Silero, CPU, бесплатно
# Меняйте SPEAKER и PHRASES, затем запускайте ячейку повторно.
# Модель хранится только во временной среде /content, Google Диск не подключается.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, urllib.request, wave, zipfile
import torch
from IPython.display import Audio, display, HTML

MODEL_ID = "v5_5_ru"
MODEL_URL = "https://models.silero.ai/models/tts/ru/v5_5_ru.pt"
SPEAKER = "xenia"  # Женские: xenia, kseniya, baya. Мужские: aidar, eugene.
SAMPLE_RATE = 48000
# Для перезаписи только выбранных фраз укажите их id: ["squat-trunk", "pushup-line"].
ONLY_IDS = []
PHRASES = [
    {"id":"voice-test", "sourceText":"Голосовые подсказки включены. Во время тренировки я озвучу замечания по технике.", "text":"Голосовые подсказки включены. Во время тренировки я подскажу, на что обратить внимание."},
    {"id":"squat-trunk", "sourceText":"Наблюдаем устойчивый наклон корпуса. Попробуйте держать корпус ровнее.", "text":"Попробуйте держать корпус ровнее."},
    {"id":"pushup-line", "sourceText":"Наблюдаем отклонение линии тела. Попробуйте выровнять плечо, таз и лодыжку.", "text":"Попробуйте выровнять плечо, таз и лодыжку."},
    {"id":"plank-horizontal", "text":"Корпус должен быть близок к горизонтальному."},
    {"id":"plank-hips-up", "text":"Поднимите таз немного выше."},
    {"id":"plank-hips-down", "text":"Опустите таз немного ниже."},
    {"id":"plank-knee", "text":"Удерживайте колено почти выпрямленным."},
    {"id":"plank-elbow", "text":"Расположите локоть под плечом, предплечье — вдоль пола."},
    {"id":"wall-sit-knee", "text":"Сверьте положение колена с примером стульчика."},
    {"id":"wall-sit-thigh", "text":"Бедро должно быть близко к горизонтальному."},
    {"id":"wall-sit-trunk", "text":"Держите корпус ближе к вертикали."},
    {"id":"wall-sit-position", "text":"Сохраняйте положение сидя у стены."},
    {"id":"bridge-legs", "text":"Обе ноги согнуты; плечи остаются в нижнем положении."},
    {"id":"bridge-position", "text":"Сохраните положение лёжа, поднимайте таз, а не корпус целиком."},
    {"id":"lunge-trunk", "text":"Старайтесь держать корпус ровнее в выпаде."},
    {"id":"lunge-support", "text":"Сохраните выбранную опорную ногу впереди."},
    {"id":"standing-trunk", "text":"Держите корпус ровнее, без выраженного раскачивания."},
    {"id":"front-hold-height", "text":"Поднимите обе руки перед собой до уровня плеч."},
    {"id":"front-hold-elbows", "text":"Держите обе руки почти выпрямленными."},
    {"id":"front-hold-position", "text":"Обе руки должны быть видны и направлены вперёд, корпус — стоя."},
    {"id":"side-bend-position", "text":"Встаньте прямо, руки с гантелями вдоль тела."},
    {"id":"side-bend-neutral", "text":"Вернитесь в нейтраль, избегайте выраженного раскачивания."},
    {"id":"arms-symmetry", "text":"Сверьте выбранное положение тела и амплитуду обеих рук."},
    {"id":"trunk-sway", "sourceText":"Наблюдаем смещение корпуса. Уменьшите раскачивание.", "text":"Уменьшите раскачивание корпуса."},
    {"id":"arms-smooth", "sourceText":"Положение рук меняется резко. Поднимайте и опускайте их плавнее.", "text":"Поднимайте и опускайте руки плавнее."},
    {"id":"lunge-support-drift", "text":"Опорная нога заметно сместилась. Сверьте выбранную сторону."},
    {"id":"bridge-shoulders", "text":"Оставьте плечи внизу; поднимайте таз, а не корпус целиком."},
    {"id":"side-bend-hips", "sourceText":"Сместился таз: старайтесь наклонять корпус без перемещения всего тела.", "text":"Старайтесь наклонять корпус без перемещения всего тела."},
]
assert len({p["id"] for p in PHRASES}) == len(PHRASES)
assert set(ONLY_IDS) <= {p["id"] for p in PHRASES}, "Неизвестный id в ONLY_IDS"
for p in PHRASES:
    p.setdefault("sourceText", p["text"])

torch.set_num_threads(2)
cache = Path("/content/playfit-silero-cache")
cache.mkdir(exist_ok=True)
model_file = cache / (MODEL_ID + ".pt")
if not model_file.exists():
    print("Загрузка официальной модели Silero. GPU не нужен.", flush=True)
    partial = model_file.with_suffix(".download")
    urllib.request.urlretrieve(MODEL_URL, partial)
    partial.rename(model_file)
model_sha256 = hashlib.file_digest(model_file.open("rb"), "sha256").hexdigest()
model = torch.package.PackageImporter(str(model_file)).load_pickle("tts_models", "model")
model.to(torch.device("cpu"))
assert SPEAKER in model.speakers, str(model.speakers)
print("Модель готова:", MODEL_ID, "Голоса:", model.speakers, flush=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = Path("/content") / ("playfit-voice-" + SPEAKER + "-" + stamp)
out.mkdir()

def synthesize(text, speaker, dest):
    with torch.inference_mode():
        audio = model.apply_tts(text=text, speaker=speaker, sample_rate=SAMPLE_RATE,
                                put_accent=True, put_yo=True).detach().cpu().flatten()
    assert audio.numel() > 0 and bool(torch.isfinite(audio).all()), "Пустой или повреждённый звук"
    peak = float(audio.abs().max())
    assert peak > 0.0001, "Тишина вместо речи"
    # Только уменьшаем слишком большой пик; без ускорения или изменения высоты голоса.
    audio = audio * min(1.0, 0.92 / peak)
    pcm = (audio.clamp(-1, 1) * 32767).round().to(torch.int16).numpy().tobytes()
    with wave.open(str(dest), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SAMPLE_RATE); w.writeframes(pcm)
    with wave.open(str(dest), "rb") as w:
        assert w.getnframes() == len(audio) and w.getnchannels() == 1
    return {"durationSeconds": round(len(audio) / SAMPLE_RATE, 3),
            "peak": round(float(audio.abs().max()), 5),
            "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(), "bytes": dest.stat().st_size}

records = []
selected = [p for p in PHRASES if not ONLY_IDS or p["id"] in ONLY_IDS]
for i, phrase in enumerate(selected, 1):
    filename = phrase["id"] + ".wav"
    info = synthesize(phrase["text"], SPEAKER, out / filename)
    records.append({**phrase, "file": filename, **info})
    print(f"{i}/{len(selected)}  {filename}  {info['durationSeconds']} с", flush=True)

preview_dir = out / "samples"
preview_dir.mkdir()
for speaker in ["xenia", "kseniya", "baya"]:
    preview_file = preview_dir / (speaker + ".wav")
    synthesize("Голосовые подсказки включены. Попробуйте держать корпус ровнее. Поднимайте и опускайте руки плавнее.", speaker, preview_file)

license_url = "https://raw.githubusercontent.com/snakers4/silero-models/master/LICENSE"
license_text = urllib.request.urlopen(license_url, timeout=30).read().decode("utf-8")
(out / "SILERO-LICENSE.txt").write_text(license_text, encoding="utf-8")
manifest = {"schemaVersion":1, "model":MODEL_ID, "modelUrl":MODEL_URL,
            "modelSha256":model_sha256, "speaker":SPEAKER, "sampleRate":SAMPLE_RATE,
            "format":"WAV PCM16 mono", "generatedAt":stamp,
            "torchVersion":torch.__version__, "pythonVersion":platform.python_version(),
            "putAccent":True, "putYo":True,
            "source":"https://github.com/snakers4/silero-models",
            "modelLicense":"CC BY-NC-SA 4.0 (see SILERO-LICENSE.txt)",
            "notes":"Учебный некоммерческий набор. Укороченные реплики связаны с исходным текстом через sourceText. Подключение к приложению выполняется отдельно.",
            "phrases":records}
(out / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(out / "phrases.json").write_text(json.dumps(PHRASES, ensure_ascii=False, indent=2), encoding="utf-8")
(out / "README.txt").write_text("""PlayFit Vision — озвучка подсказок.
Generated with Silero Models by Silero AI Team.
Source: https://github.com/snakers4/silero-models
Model license: CC BY-NC-SA 4.0; see SILERO-LICENSE.txt.
Учебное некоммерческое использование. Для коммерческого использования нужно отдельно проверить права.
Файлы .wav — готовые записи. manifest.json связывает исходные и произнесённые тексты.
Модель в архив не включена. Для изменений сохраните блокнот и измените PHRASES/SPEAKER/ONLY_IDS.
""", encoding="utf-8")
archive = out.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for file in sorted(out.rglob("*")):
        if file.is_file(): z.write(file, file.relative_to(out))
print("ГОТОВО:", len(records), "фраз; архив:", archive.name, "размер:", archive.stat().st_size, "байт", flush=True)
print("Речь проверена на непустые данные, конечные значения, длительность и пики. Произношение оцените прослушиванием.")
display(HTML("<h3>Образцы женских голосов</h3>"))
for speaker in ["xenia", "kseniya", "baya"]:
    print(speaker)
    display(Audio(filename=str(preview_dir / (speaker + ".wav"))))
print("Основной набор:", SPEAKER)


In [ ]:
#@title 2. Скачать готовый архив
from google.colab import files
assert archive.is_file(), "Сначала выполните ячейку 1"
files.download(str(archive))


# Как пользоваться

1. Выполните ячейку **1** в обычной среде CPU — подписка и GPU не нужны.
2. Прослушайте образцы Xenia, Kseniya и Baya. Основной набор по умолчанию: **xenia**. Для другого голоса измените `SPEAKER` и повторите генерацию.
3. Выполните ячейку **2** и сохраните ZIP на компьютер. В архиве: 28 WAV, образцы, тексты, манифест и лицензия.
4. Для изменения реплики отредактируйте `PHRASES`; для отдельных файлов задайте `ONLY_IDS`, например `["squat-trunk"]`. Каждое выполнение создаёт новый архив, старые не стираются. При изменении текста в приложении обновляйте также `sourceText`.

## После скачивания

**Сохраните этот блокнот.** Код и список фраз пригодятся для изменений. Модель находится только во временной среде `/content/playfit-silero-cache`; Google Диск не подключается. После проверки скачанного архива выберите **Среда выполнения → Отключиться и удалить среду выполнения**. Это удалит модель, временные WAV и установленные в среде зависимости, но сохранит блокнот на Диске и скачанный архив. Позже запустите ячейки заново: модель загрузится снова.

Генератор использует [Silero Models](https://github.com/snakers4/silero-models), модель `v5_5_ru`. Лицензия модели: **CC BY-NC-SA 4.0**, полный текст включён в архив. Набор подготовлен для учебного некоммерческого проекта.

Автоматические проверки подтверждают структуру WAV, непустые данные, длительность и отсутствие переполнения. Естественность и произношение нужно оценить прослушиванием. Записи ещё нужно подключить к проигрывателю приложения.


## Если Colab не показывает плееры

Откройте слева **Файлы**, нажмите **Обновить**, затем правой кнопкой на `playfit-voice-*.zip` → **Скачать**. В этом запуске архив скачан именно так. WAV можно прослушать на компьютере; сбой интерфейса вывода не мешает генерации.
